In [1]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
from dash import ctx # Used for button trigger callback
import base64
import re

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#Import CRUD operation
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

#Requests all documents be returned from AAC mongo-database to a Panda-database-structure
df = pd.DataFrame.from_records(db.read({}))

#Drops _id column from database to prevent TypeErrors 
df.drop(columns=['_id'],inplace=True)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#Loads and Formats the image 
image_filename = 'Grazioso_Salvare_Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Stores image with an anchor to SNHU Website as img
img =html.A(href='https://www.snhu.edu/',children=[
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),height = 200,title='https://www.snhu.edu/')])

# Formats the application layout
app.layout = html.Div([
    
    #----------------------------------------Icon and Title Setup-----------------------------------------
    
    html.Hr(),
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            img, # Grazioso Icon 
            ),
        html.Div([
            html.Center(html.B(html.H1(
                'Grazioso Salvare',
                 style = {'margin-left': '10px',
                          'margin-bottom': '1px',
                          'font-family': 'Didot, serif',
                          'font-size' : '100px',
                          'color' : '#CA1450',                                                                  
                    })
                )
            )
        ])
    ]),
    html.B('Search and Rescue Training',
           style={
               'margin-left': '450px',
               'margin-top': '1px',
               'font-family': 'Didot, serif',
               'font-size' : '30px',
               'color' : '#CA1450',
           }
    ),
    html.Div('Developed by Andrew Nelson', style={'font-size': '10px','margin-left':'600px'}),    
    html.Hr(),
    
    #----------------------------------------Button Setup-----------------------------------------
    html.Div(
        className = 'buttonRow',
        style = {'display':'flex', 'margin-left':'250px'},
        children = [
            html.Button(id = 'button-1', n_clicks = 0, children = 'Water Rescue',
                        style = {'margin':'10px','padding':'10px 15px' }),
            html.Button(id = 'button-2', n_clicks = 0, children = 'Mountain or Wilderness Rescue',
                        style = {'margin':'10px','padding':'10px 15px' }),
            html.Button(id = 'button-3', n_clicks = 0, children = 'Disaster Rescue or Individual Tracking',
                        style = {'margin':'10px','padding':'10px 15px' }),
            html.Button(id = 'button-4', n_clicks = 0, children = 'RESET',
                        style = {'margin':'10px','padding':'10px 15px' }),
            
        ]
    
    ),
    html.Hr(),
    html.Br(),
    
    #------------------------------Data Table Initialize and Setup---------------------------------
    
    html.Center(dash_table.DataTable(id='datatable-id',
         data=df.to_dict('records'),
         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
         fixed_rows={'headers': True, 'data':0}, #Keeps column header frozen during scrolling
         row_selectable='single', #Enables row selections used in geomap
         fill_width = False,
         selected_rows = [0],
         column_selectable = 'single',
         style_table={'overflowX': 'auto', 'overflowY':'auto', 'height':'300px'},                         
         style_cell={
             'minWidth': '180px', 'width': '180px', 'maxWidth': '180px',
             'overflow': 'hidden',
             'textOverflow': 'ellipsis',
             'textAlign': 'center'
         },
         style_header={
             'backgroundColor': 'rgb(30, 30, 30)',
             'color': 'white',
             'fontWeight' : 'bold',
             'border'  : '2px solid #CA1450'
                        },
         style_data={
             'backgroundColor': 'rgb(50, 50, 50)',
             'color': 'white',
             'border'  : '2px solid #CA1450'
         },
         style_data_conditional=[],
         )
    ),
        html.Div('Developed by Andrew Nelson', style={'font-size': '10px','margin-left':'600px'}),
    #--------------------------------Map and Pie Chart Layout Configuration-----------------------
    
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6'
            )
        ]),    
])


#############################################
# Interaction Between Components / Controller
#############################################


#-------------------------Button Callbacks to Filter Rescue Dog Categories-------------------
    
@app.callback(
    Output('datatable-id','data'),
    [Input('button-1', 'n_clicks'),
     Input('button-2', 'n_clicks'),
     Input('button-3', 'n_clicks'),
     Input('button-4', 'n_clicks'),
    ])

def update_dashTable(button1,button2,button3,button4):
    
    # Start Case data set with a copy
    dff = pd.DataFrame.from_records(db.read({})) 

    # Button 1 for Water Rescue filter 
    if 'button-1' == ctx.triggered_id:

        dff = pd.DataFrame.from_records(db.read(
        {'$or': [
            {'breed' : {'$regex':'.*Lab.*'}},
            {'breed' : {'$regex':'.*Chesa.*'}},
            {'breed' : {'$regex':'.*Newf.*'}},
        ],
         'sex_upon_outcome':'Intact Female',
         'age_upon_outcome_in_weeks':{'$gte':26,'$lte':156}
        }
        ))
    # Button 2 for Mountain or Wilderness Rescue filter 
    elif 'button-2' == ctx.triggered_id:
        dff = pd.DataFrame.from_records(db.read(
        {'$or': [
            {'breed' : {'$regex':'.*Germa.*'}},
            {'breed' : {'$regex':'.*Alaskan Mal.*'}},
            {'breed' : {'$regex':'.*Old English.*'}},
            {'breed' : {'$regex':'.*Husky.*'}},
            {'breed' : {'$regex':'.*Rott.*'}},
        ],
         'sex_upon_outcome':'Intact Male',
         'age_upon_outcome_in_weeks':{'$gte':26,'$lte':156}
        }
        ))

    # Button 3 for Disaster Rescue or Individual Tracking filter 
    elif 'button-3' == ctx.triggered_id:
        dff = pd.DataFrame.from_records(db.read(
            {'$or': [
            {'breed' : {'$regex':'.*Doberman P.*'}},
            {'breed' : {'$regex':'.*Germa.*'}},
            {'breed' : {'$regex':'.*Gold.*'}},
            {'breed' : {'$regex':'.*Blood.*'}},
            {'breed' : {'$regex':'.*Rott.*'}},
        ],
         'sex_upon_outcome':'Intact Male',
         'age_upon_outcome_in_weeks':{'$gte':20,'$lte':300}
        }
        ))
        
    # Button 4 for RESET to clear filter    
    elif 'button-4' == ctx.triggered_id:
        dff = pd.DataFrame.from_records(db.read({}))
        #print(dff)    
    
    #Drops _id column for outgoing data
    dff.drop(columns=['_id'],inplace = True)
    
    #Data to be returned in callback
    return dff.to_dict('records')

#-----------------------------Callback for Pie Chart Setup--------------------------------   
    
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])

#Update graph based on filter button
def update_graphs(viewData):
    
    # Start Case data set with a copy
    dffPie = pd.DataFrame.from_records(viewData)
    #Create pie chart using breed as sorting aspect
    fig = px.pie(dffPie, names='breed', title='Preferred Animals')
    #Formats the pie chart
    fig.update_traces(textposition='inside',textinfo = 'percent+label')

    #Returns updated graph figure
    return [
        dcc.Graph(
             figure = fig
        )    
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        raise no_update
    else:
        return [{
            'if': { 'column_id': i },
            'background_color': 'Blue'
        }
        for i in selected_columns]


#------------------------------------Geo Map Setup----------------------------------------------
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):

    if viewData is None:
        return 
    elif index is None:
        return 
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]

    # Returns a map with a marker of the current Animal selected from the row selectable option    
    return [
        dl.Map(style={'width': '500px', 'height': '500px'}, center=[dff.iloc[row,13],dff.iloc[row,14]], zoom=9, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]

app.run_server(debug=True)


Dash app running on http://127.0.0.1:11534/
